<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES_Stage6D_Cell_6D_1C1_Editorial_Comment4_Reproducibility_Completion_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

# Cell 1 — Setup
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, platform, subprocess, sys
import numpy as np, pandas as pd, scipy, sklearn, joblib, matplotlib, pyarrow
ROOT=Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
OUTDIR=ROOT/'outputs/revision_support/stage6d_1c1_comment4_reproducibility_v1'
OUTDIR.mkdir(parents=True,exist_ok=True)
T0=ROOT/'data_interim/t0_rcv_target_genes_corrected_v1_2.parquet'
T1=ROOT/'data_interim/t1_rcv_target_genes_harmonized_v1.parquet'
EXPECTED_T0_SHA256='f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d'
EXPECTED_T1_SHA256='5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c'
REPO_URL='https://github.com/SANGHATI23/genomic-evidence-reliability.git'
ZENODO_DOI=''  # Fill only after creating a real archival DOI.
def sha256_file(path,chunk_size=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk_size)
            if not b: break
            h.update(b)
    return h.hexdigest()
for p in [T0,T1]:
    if not p.exists(): raise FileNotFoundError(p)
print('Output:',OUTDIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output: /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/revision_support/stage6d_1c1_comment4_reproducibility_v1


In [5]:

# Cell 2 — Fail-closed T0/T1 verification
t0_hash=sha256_file(T0); t1_hash=sha256_file(T1)
print('T0 SHA-256:',t0_hash); print('T1 SHA-256:',t1_hash)
if t0_hash!=EXPECTED_T0_SHA256: raise RuntimeError('T0 frozen Parquet hash mismatch')
if t1_hash!=EXPECTED_T1_SHA256: raise RuntimeError('T1 frozen Parquet hash mismatch')
print('PASS — frozen T0/T1 artifacts verified.')


T0 SHA-256: f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d
T1 SHA-256: 5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c
PASS — frozen T0/T1 artifacts verified.


In [6]:
# Cell 3 — Derive canonical gene column from target_genes_json
# and identify the classification-group column

import json
import pandas as pd

# t0_all and t1_all should already be loaded from Cell 2

def parse_single_target_gene(value):
    """
    Parse target_genes_json and return the single normalized target gene.
    Fail closed if the row does not contain exactly one gene.
    """
    if pd.isna(value):
        return pd.NA

    if isinstance(value, list):
        genes = value
    else:
        try:
            genes = json.loads(str(value))
        except (json.JSONDecodeError, TypeError, ValueError):
            return pd.NA

    if not isinstance(genes, list):
        return pd.NA

    genes = sorted({
        str(g).strip().upper()
        for g in genes
        if g is not None and str(g).strip()
    })

    if len(genes) != 1:
        return pd.NA

    return genes[0]


# ------------------------------------------------------------
# 1. Confirm the actual source columns exist
# ------------------------------------------------------------

required_t0 = {
    "target_genes_json",
    "aggregate_classification_group",
}

required_t1 = {
    "target_genes_json",
    "aggregate_classification_group",
}

missing_t0 = required_t0 - set(t0_all.columns)
missing_t1 = required_t1 - set(t1_all.columns)

if missing_t0:
    raise KeyError(f"T0 missing required columns: {sorted(missing_t0)}")

if missing_t1:
    raise KeyError(f"T1 missing required columns: {sorted(missing_t1)}")


# ------------------------------------------------------------
# 2. Derive target_gene exactly as done in the original workflow
# ------------------------------------------------------------

t0_all["target_gene"] = (
    t0_all["target_genes_json"]
    .apply(parse_single_target_gene)
)

t1_all["target_gene"] = (
    t1_all["target_genes_json"]
    .apply(parse_single_target_gene)
)


# ------------------------------------------------------------
# 3. Fail closed on malformed / multi-gene records
# ------------------------------------------------------------

t0_missing_gene = int(t0_all["target_gene"].isna().sum())
t1_missing_gene = int(t1_all["target_gene"].isna().sum())

print("T0 rows without exactly one target gene:", t0_missing_gene)
print("T1 rows without exactly one target gene:", t1_missing_gene)

if t0_missing_gene != 0:
    raise RuntimeError(
        f"T0 contains {t0_missing_gene:,} rows without exactly one target gene."
    )

if t1_missing_gene != 0:
    raise RuntimeError(
        f"T1 contains {t1_missing_gene:,} rows without exactly one target gene."
    )


# ------------------------------------------------------------
# 4. Set the canonical columns for the remaining notebook
# ------------------------------------------------------------

T0_GENE = "target_gene"
T1_GENE = "target_gene"

T0_CLASS = "aggregate_classification_group"
T1_CLASS = "aggregate_classification_group"


# ------------------------------------------------------------
# 5. Verify expected target-gene set
# ------------------------------------------------------------

EXPECTED_GENES = {"BRCA1", "BRCA2", "MLH1", "EGFR"}

observed_t0_genes = set(t0_all[T0_GENE].dropna().unique())
observed_t1_genes = set(t1_all[T1_GENE].dropna().unique())

print("\nT0 genes:", sorted(observed_t0_genes))
print("T1 genes:", sorted(observed_t1_genes))

if observed_t0_genes != EXPECTED_GENES:
    raise RuntimeError(
        f"Unexpected T0 target-gene set: {observed_t0_genes}"
    )

if observed_t1_genes != EXPECTED_GENES:
    raise RuntimeError(
        f"Unexpected T1 target-gene set: {observed_t1_genes}"
    )


# ------------------------------------------------------------
# 6. Display counts before continuing
# ------------------------------------------------------------

print("\nT0 gene counts")
print(
    t0_all[T0_GENE]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nT1 gene counts")
print(
    t1_all[T1_GENE]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nT0 classification groups")
print(
    t0_all[T0_CLASS]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

print("\nT1 classification groups")
print(
    t1_all[T1_CLASS]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

print("\nPASS — canonical gene and classification columns established.")


T0 rows without exactly one target gene: 0
T1 rows without exactly one target gene: 0

T0 genes: ['BRCA1', 'BRCA2', 'EGFR', 'MLH1']
T1 genes: ['BRCA1', 'BRCA2', 'EGFR', 'MLH1']

T0 gene counts
target_gene
BRCA1    25408
BRCA2    34915
EGFR      2400
MLH1      8936

T1 gene counts
target_gene
BRCA1    32603
BRCA2    49221
EGFR      5412
MLH1     13684

T0 classification groups
aggregate_classification_group
Benign/Likely benign            19483
Conflicting                      1480
Other                            3961
Pathogenic/Likely pathogenic    20430
VUS                             26305

T1 classification groups
aggregate_classification_group
Benign/Likely benign            30481
Conflicting                      6572
Missing                           597
Other                            3450
Pathogenic/Likely pathogenic    27299
VUS                             32521

PASS — canonical gene and classification columns established.


In [7]:

# Cell 4 — Exact counts by gene and aggregate classification group
t0=t0_all[[T0_GENE,T0_CLASS]].copy(); t1=t1_all[[T1_GENE,T1_CLASS]].copy()
if len(t0)!=71659 or len(t1)!=100920: raise RuntimeError('Unexpected frozen row count')

def ctab(df,gene_col,class_col,tp):
    x=(df.groupby([gene_col,class_col],dropna=False).size().rename('record_count').reset_index().rename(columns={gene_col:'gene',class_col:'classification_group'}))
    x.insert(0,'timepoint',tp)
    return x.sort_values(['gene','classification_group']).reset_index(drop=True)

gene_class=pd.concat([ctab(t0,T0_GENE,T0_CLASS,'T0'),ctab(t1,T1_GENE,T1_CLASS,'T1')],ignore_index=True)
gene_totals=pd.concat([
 t0[T0_GENE].value_counts().rename_axis('gene').reset_index(name='record_count').assign(timepoint='T0'),
 t1[T1_GENE].value_counts().rename_axis('gene').reset_index(name='record_count').assign(timepoint='T1')],ignore_index=True)[['timepoint','gene','record_count']].sort_values(['timepoint','gene'])
class_totals=pd.concat([
 t0[T0_CLASS].value_counts().rename_axis('classification_group').reset_index(name='record_count').assign(timepoint='T0'),
 t1[T1_CLASS].value_counts().rename_axis('classification_group').reset_index(name='record_count').assign(timepoint='T1')],ignore_index=True)[['timepoint','classification_group','record_count']].sort_values(['timepoint','classification_group'])

expected_gene={'BRCA1':25408,'BRCA2':34915,'EGFR':2400,'MLH1':8936}
obs_gene=dict(zip(gene_totals.query("timepoint=='T0'")['gene'],gene_totals.query("timepoint=='T0'")['record_count'].astype(int)))
if obs_gene!=expected_gene: raise RuntimeError(obs_gene)
expected_class={'VUS':26305,'Pathogenic/Likely pathogenic':20430,'Benign/Likely benign':19483,'Other':3961,'Conflicting':1480}
obs_class=dict(zip(class_totals.query("timepoint=='T0'")['classification_group'],class_totals.query("timepoint=='T0'")['record_count'].astype(int)))
if obs_class!=expected_class: raise RuntimeError(obs_class)

gene_class_path=OUTDIR/'stage6d_1c1_gene_by_class_counts_T0_T1_v1.csv'; gene_class.to_csv(gene_class_path,index=False)
gene_totals_path=OUTDIR/'stage6d_1c1_gene_totals_T0_T1_v1.csv'; gene_totals.to_csv(gene_totals_path,index=False)
class_totals_path=OUTDIR/'stage6d_1c1_class_totals_T0_T1_v1.csv'; class_totals.to_csv(class_totals_path,index=False)
print('T0 gene totals'); display(gene_totals.query("timepoint=='T0'"))
print('T0 class totals'); display(class_totals.query("timepoint=='T0'"))
print('T0/T1 gene × class counts'); display(gene_class)


T0 gene totals


,timepoint,gene,record_count
1,T0,BRCA1,25408
0,T0,BRCA2,34915
3,T0,EGFR,2400
2,T0,MLH1,8936


T0 class totals


,timepoint,classification_group,record_count
2,T0,Benign/Likely benign,19483
4,T0,Conflicting,1480
3,T0,Other,3961
1,T0,Pathogenic/Likely pathogenic,20430
0,T0,VUS,26305


T0/T1 gene × class counts


,timepoint,gene,classification_group,record_count
0,T0,BRCA1,Benign/Likely benign,6404
1,T0,BRCA1,Conflicting,454
2,T0,BRCA1,Other,3582
3,T0,BRCA1,Pathogenic/Likely pathogenic,7638
4,T0,BRCA1,VUS,7330
5,T0,BRCA2,Benign/Likely benign,9744
6,T0,BRCA2,Conflicting,879
7,T0,BRCA2,Other,233
8,T0,BRCA2,Pathogenic/Likely pathogenic,9719
9,T0,BRCA2,VUS,14340


In [8]:

# Cell 5 — Exact source ledger
source_ledger=pd.DataFrame([
 {'timepoint':'T0','release_label':'2023-01','archive_publication_date':'2023-01-05','embedded_data_cutoff':'2022-12-31','source_filename':'ClinVarFullRelease_2023-01.xml.gz','archive_url':'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/RCV_xml_old_format/archive/2023/ClinVarFullRelease_2023-01.xml.gz','source_sha256':'911c8a58872ea89cc7bb4f1ee3362596d965f5103422b30f456abaf99c41e5e7','frozen_parquet':T0.name,'frozen_parquet_sha256':t0_hash,'retained_rcvs':len(t0)},
 {'timepoint':'T1','release_label':'2026-01','archive_publication_date':'2026-01-01','embedded_data_cutoff':'2025-12-27','source_filename':'ClinVarRCVRelease_2026-01.xml.gz','archive_url':'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/RCV_release/ClinVarRCVRelease_2026-01.xml.gz','source_sha256':'3fba206f1e3086306472ab7b0ae324d4ae516da84857038d03f2937fd20a6e55','frozen_parquet':T1.name,'frozen_parquet_sha256':t1_hash,'retained_rcvs':len(t1)}])
source_path=OUTDIR/'stage6d_1c1_exact_clinvar_source_ledger_v1.csv'; source_ledger.to_csv(source_path,index=False); display(source_ledger)


,timepoint,release_label,archive_publication_date,embedded_data_cutoff,source_filename,archive_url,source_sha256,frozen_parquet,frozen_parquet_sha256,retained_rcvs
0,T0,2023-01,2023-01-05,2022-12-31,ClinVarFullRelease_2023-01.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,911c8a58872ea89cc7bb4f1ee3362596d965f5103422b3...,t0_rcv_target_genes_corrected_v1_2.parquet,f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2...,71659
1,T1,2026-01,2026-01-01,2025-12-27,ClinVarRCVRelease_2026-01.xml.gz,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,3fba206f1e3086306472ab7b0ae324d4ae516da8485703...,t1_rcv_target_genes_harmonized_v1.parquet,5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b1...,100920


In [9]:

# Cell 6 — Exact extraction/filter rules
filters=[
 [1,'Primary unit','Condition-specific ClinVar RCV record (variant-condition relationship), not a VariationID-level aggregate.'],
 [2,'Target genes','Retain RCVs mapped to BRCA1, BRCA2, MLH1, or EGFR.'],
 [3,'Study scope','BRCA1, BRCA2, and MLH1 are primary genes; EGFR is a prespecified exploratory somatic-oncology stratum.'],
 [4,'Gene assignment','Require exactly one normalized target gene from the four-gene set; zero, multiple, or unexpected assignments fail QC.'],
 [5,'RCV identity','Require a valid unique RCV accession.'],
 [6,'Classification','Preserve aggregate ClinVar classification and harmonize it to project classification groups; future T1 classification is not used for model construction.'],
 [7,'Temporal separation','Construct GES from T0 metadata only; no T1 fitting, recalibration, threshold optimization, or proxy-label construction.'],
 [8,'Temporal linkage','Use exact RCV continuity first; accept only conservative one-to-one continuity links supported by VariationID, VCV accession, target gene, condition overlap, and unique candidate topology.'],
 [9,'Unresolved linkage','Censor unresolved/non-unique relationships rather than treating them as stable.'],
 [10,'Outcome evaluability','Evaluate only interpretable future classifications on the prespecified primary axis; nonevaluable axes/nonprimary classifications are not assigned negative labels.']]
filters_df=pd.DataFrame(filters,columns=['step','rule','exact_definition']); filters_path=OUTDIR/'stage6d_1c1_extraction_filter_rules_v1.csv'; filters_df.to_csv(filters_path,index=False); display(filters_df)


,step,rule,exact_definition
0,1,Primary unit,Condition-specific ClinVar RCV record (variant...
1,2,Target genes,"Retain RCVs mapped to BRCA1, BRCA2, MLH1, or E..."
2,3,Study scope,"BRCA1, BRCA2, and MLH1 are primary genes; EGFR..."
3,4,Gene assignment,Require exactly one normalized target gene fro...
4,5,RCV identity,Require a valid unique RCV accession.
5,6,Classification,Preserve aggregate ClinVar classification and ...
6,7,Temporal separation,Construct GES from T0 metadata only; no T1 fit...
7,8,Temporal linkage,Use exact RCV continuity first; accept only co...
8,9,Unresolved linkage,Censor unresolved/non-unique relationships rat...
9,10,Outcome evaluability,Evaluate only interpretable future classificat...


In [10]:

# Cell 7 — Frozen Git commit and exact software environment
def remote_head(url):
    s=subprocess.check_output(['git','ls-remote',url,'HEAD'],text=True,timeout=30).strip(); return s.split()[0]
git_head=remote_head(REPO_URL)
env={'Python':sys.version.split()[0],'Platform':platform.platform(),'NumPy':np.__version__,'pandas':pd.__version__,'SciPy':scipy.__version__,'scikit-learn':sklearn.__version__,'joblib':joblib.__version__,'matplotlib':matplotlib.__version__,'pyarrow':pyarrow.__version__}
env_df=pd.DataFrame([env]); env_path=OUTDIR/'stage6d_1c1_software_environment_v1.csv'; env_df.to_csv(env_path,index=False)
requirements_path=OUTDIR/'requirements_springer_reproducibility_v1.txt'; requirements_path.write_text('\n'.join([f'numpy=={np.__version__}',f'pandas=={pd.__version__}',f'scipy=={scipy.__version__}',f'scikit-learn=={sklearn.__version__}',f'joblib=={joblib.__version__}',f'matplotlib=={matplotlib.__version__}',f'pyarrow=={pyarrow.__version__}'])+'\n',encoding='utf-8')
print('Repository HEAD:',git_head); display(env_df)


Repository HEAD: 9169beca4bd9ebe535def60e98b4a885b95f51ad


,Python,Platform,NumPy,pandas,SciPy,scikit-learn,joblib,matplotlib,pyarrow
0,3.13.15,Linux-6.6.122+-x86_64-with-glibc2.35,2.1.3,2.2.3,1.16.3,1.6.1,1.5.3,3.10.0,18.1.0


In [11]:

# Cell 8 — Manuscript-ready reproducibility table
rows=[
 ['T0 ClinVar source','ClinVarFullRelease_2023-01.xml.gz; archive publication 2023-01-05; embedded cutoff 2022-12-31'],
 ['T1 ClinVar source','ClinVarRCVRelease_2026-01.xml.gz; archive publication 2026-01-01; embedded cutoff 2025-12-27'],
 ['Primary unit','Condition-specific RCV record'],
 ['Genes','BRCA1, BRCA2, MLH1 (primary); EGFR (exploratory)'],
 ['T0 retained RCVs','71,659'],['T1 retained RCVs','100,920'],['Locked evaluable temporal cohort','66,636'],['Future-instability events','6,485'],
 ['Frozen repository commit',git_head],['T0 Parquet SHA-256',t0_hash],['T1 Parquet SHA-256',t1_hash],
 ['Software','; '.join(f'{k} {v}' for k,v in env.items() if k!='Platform')]]
repro=pd.DataFrame(rows,columns=['Item','Exact reproducibility information']); repro_path=OUTDIR/'stage6d_1c1_manuscript_reproducibility_table_v1.csv'; repro.to_csv(repro_path,index=False); display(repro)


,Item,Exact reproducibility information
0,T0 ClinVar source,ClinVarFullRelease_2023-01.xml.gz; archive pub...
1,T1 ClinVar source,ClinVarRCVRelease_2026-01.xml.gz; archive publ...
2,Primary unit,Condition-specific RCV record
3,Genes,"BRCA1, BRCA2, MLH1 (primary); EGFR (exploratory)"
4,T0 retained RCVs,"71,659"
5,T1 retained RCVs,"100,920"
6,Locked evaluable temporal cohort,"66,636"
7,Future-instability events,"6,485"
8,Frozen repository commit,9169beca4bd9ebe535def60e98b4a885b95f51ad
9,T0 Parquet SHA-256,f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2...


In [12]:

# Cell 9 — Permanent data/code availability text
if ZENODO_DOI.strip():
    code_archive='A versioned archival copy of the code and reproducibility package is permanently available through Zenodo at https://doi.org/'+ZENODO_DOI.strip()+'.'; archive_status='PASS_ARCHIVAL_DOI_PRESENT'
else:
    code_archive='A permanent archival DOI has not yet been entered. Create the GitHub/Zenodo release before resubmission.'; archive_status='PENDING_ARCHIVAL_DOI'
data_text=('Data availability. The study used public ClinVar RCV XML archives from NCBI. The frozen baseline source was ClinVarFullRelease_2023-01.xml.gz (archive publication 5 January 2023; embedded data cutoff 31 December 2022), and the follow-up source was ClinVarRCVRelease_2026-01.xml.gz (archive publication 1 January 2026; embedded data cutoff 27 December 2025). The source files remain available through the NCBI ClinVar archive. Exact source identifiers, checksums, extraction rules, and cohort counts are provided in the reproducibility package.')
code_text='Code availability. Analysis code is available in the public GitHub repository at frozen commit '+git_head+'. '+code_archive
avail_path=OUTDIR/'stage6d_1c1_data_code_availability_v1.txt'; avail_path.write_text(data_text+'\n\n'+code_text+'\n',encoding='utf-8')
print(data_text); print(); print(code_text); print('\nStatus:',archive_status)


Data availability. The study used public ClinVar RCV XML archives from NCBI. The frozen baseline source was ClinVarFullRelease_2023-01.xml.gz (archive publication 5 January 2023; embedded data cutoff 31 December 2022), and the follow-up source was ClinVarRCVRelease_2026-01.xml.gz (archive publication 1 January 2026; embedded data cutoff 27 December 2025). The source files remain available through the NCBI ClinVar archive. Exact source identifiers, checksums, extraction rules, and cohort counts are provided in the reproducibility package.

Code availability. Analysis code is available in the public GitHub repository at frozen commit 9169beca4bd9ebe535def60e98b4a885b95f51ad. A permanent archival DOI has not yet been entered. Create the GitHub/Zenodo release before resubmission.

Status: PENDING_ARCHIVAL_DOI


In [13]:

# Cell 10 — Comment 4 response draft
parts=[]
parts.append('COMMENT 4 — Reproducibility')
parts.append('')
parts.append('Response:')
parts.append('We expanded the reproducibility section and added a dedicated source/cohort table. The revised manuscript now identifies the exact ClinVar RCV releases used for the temporal study: ClinVarFullRelease_2023-01.xml.gz for T0 (archive publication date 5 January 2023; embedded data cutoff 31 December 2022) and ClinVarRCVRelease_2026-01.xml.gz for T1 (archive publication date 1 January 2026; embedded data cutoff 27 December 2025). We also report archived source locations and SHA-256 checksums.')
parts.append('')
parts.append('We now state the extraction rules explicitly. The primary unit is the condition-specific RCV record. We retained records mapped to BRCA1, BRCA2, MLH1, or EGFR, required exactly one normalized target-gene assignment, preserved aggregate classification metadata, and treated EGFR as a prespecified exploratory stratum. T0-to-T1 linkage used exact RCV continuity plus only conservative one-to-one continuity links; unresolved relationships were censored rather than treated as stable. No T1 information was used to fit, recalibrate, or threshold-optimize GES.')
parts.append('')
parts.append('The revised reproducibility package reports record counts by gene and aggregate classification group for T0 and T1, in addition to the overall T0 count of 71,659 RCVs, T1 count of 100,920 RCVs, and the locked evaluable cohort of 66,636 RCVs.')
parts.append('')
parts.append('Frozen repository commit: '+git_head)
parts.append('Software environment: '+'; '.join(f'{k} {v}' for k,v in env.items()))
parts.append('Permanent archive status: '+archive_status)
response='\n'.join(parts)
response_path=OUTDIR/'stage6d_1c1_comment4_response_draft_v1.txt'; response_path.write_text(response+'\n',encoding='utf-8'); print(response)


COMMENT 4 — Reproducibility

Response:
We expanded the reproducibility section and added a dedicated source/cohort table. The revised manuscript now identifies the exact ClinVar RCV releases used for the temporal study: ClinVarFullRelease_2023-01.xml.gz for T0 (archive publication date 5 January 2023; embedded data cutoff 31 December 2022) and ClinVarRCVRelease_2026-01.xml.gz for T1 (archive publication date 1 January 2026; embedded data cutoff 27 December 2025). We also report archived source locations and SHA-256 checksums.

We now state the extraction rules explicitly. The primary unit is the condition-specific RCV record. We retained records mapped to BRCA1, BRCA2, MLH1, or EGFR, required exactly one normalized target-gene assignment, preserved aggregate classification metadata, and treated EGFR as a prespecified exploratory stratum. T0-to-T1 linkage used exact RCV continuity plus only conservative one-to-one continuity links; unresolved relationships were censored rather than trea

In [14]:

# Cell 11 — Final manifest and readiness gate
artifacts=[gene_class_path,gene_totals_path,class_totals_path,source_path,filters_path,env_path,requirements_path,repro_path,avail_path,response_path]
manifest=[]
for p in artifacts:
    p=Path(p); manifest.append({'artifact':str(p),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
manifest=pd.DataFrame(manifest); manifest_path=OUTDIR/'stage6d_1c1_manifest_v1.csv'; manifest.to_csv(manifest_path,index=False); display(manifest)
print('COMMENT 4 READINESS')
print('Exact ClinVar release/date: PASS')
print('Extraction filters: PASS')
print('Counts by gene and class: PASS')
print('Repository commit: PASS')
print('Software environment: PASS')
print('Permanent data archive: PASS (NCBI archive)')
print('Permanent code archive:',archive_status)
if archive_status=='PASS_ARCHIVAL_DOI_PRESENT': print('\nPASS — Editorial Comment 4 COMPLETE.')
else: print('\nPENDING ONLY ZENODO DOI. No new scientific analysis is required.')


,artifact,bytes,sha256
0,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,1227,4d05428571ddfc24edeb35d5da7b573f714d839b66b776...
1,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,141,ef9e32f0a61057b18df584500a6eba6179eb61a89441b2...
2,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,289,5956bae342ca69bbe439932c55e3c95cf1caf341abd8e9...
3,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,855,dcb4688ebb208493343a5f89d7c943abe83a7df2c992e0...
4,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,1343,01a31525ada09bfad398bc4cbb7700872340b0c67fc65a...
5,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,164,dac607b86b49c35e9c9a759724a1bd95c740d25010afe2...
6,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,110,a3285b3ca19c0d5ba94f72318e931b491f883a77bb3b1c...
7,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,859,791da640cd22fdc2b54fee6ee3e9d2fa125f1bf0240a17...
8,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,787,42256f149f751cc79ab0997c9cba86e6cdb998a91732e9...
9,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,1644,4a14741a8dbe6cbf9055536b473cec816eae6b092ca5ed...


COMMENT 4 READINESS
Exact ClinVar release/date: PASS
Extraction filters: PASS
Counts by gene and class: PASS
Repository commit: PASS
Software environment: PASS
Permanent data archive: PASS (NCBI archive)
Permanent code archive: PENDING_ARCHIVAL_DOI

PENDING ONLY ZENODO DOI. No new scientific analysis is required.
